# Module 01: Azure Data Factory Fundamentals & Python SDK Integration

> **Data Engineering Diaries** | Master Cloud-Native Data Integration with Azure Data Factory (ADF)

---

## Overview

Azure Data Factory (ADF) is Microsoft's serverless data integration and orchestration platform. It allows data engineers to construct ETL/ELT workflows that ingest, transform, and move data across multi-cloud and on-premise sources.

### Core Concepts Hierarchy
```
Azure Data Factory
 ├── Linked Services      -> Connection strings & credentials (Blob, ADLS, SQL, Snowflake)
 ├── Datasets             -> Data schema & location pointers (Parquet, CSV, JSON, Relational)
 ├── Pipelines            -> Sequential/Parallel workflows containing Activities
 │    └── Activities      -> Execution units (Copy, DataFlow, Lookup, ForEach, IfCondition)
 ├── Integration Runtimes -> Compute infrastructure (Azure IR, Self-Hosted IR, SSIS IR)
 └── Triggers             -> Automated schedulers (Schedule, Tumbling Window, Storage Event)
```

In this notebook, we demonstrate how to programmatically manage and execute ADF resources using the Python `azure-mgmt-datafactory` SDK.

In [ ]:
import sys
import os
from pathlib import Path

# Add project root to Python path
sys.path.append(str(Path.cwd().parent))

from src.adf_client import ADFClient, ADFConfig
from src.linked_services import LinkedServiceManager
from src.datasets import DatasetManager
from src.pipelines.copy_activity_pipeline import CopyActivityPipelineManager
from src.monitoring import PipelineMonitor

# Initialize ADFClient in Mock Mode for offline demo execution
config = ADFConfig(
    tenant_id="00000000-0000-0000-0000-000000000000",
    client_id="00000000-0000-0000-0000-000000000000",
    client_secret="mock-secret",
    subscription_id="00000000-0000-0000-0000-000000000000",
    resource_group="rg-data-engineering-dev",
    factory_name="adf-data-engineering-dev",
    mock_mode=True
)

client = ADFClient(config)
print("ADFClient Initialized:", client)

## Step 1: Provisioning Linked Services & Datasets

We deploy:
1. Azure Key Vault Linked Service for secret references.
2. ADLS Gen2 Linked Service using System-Assigned Managed Identity.
3. Source CSV & Target Parquet Datasets.

In [ ]:
ls_mgr = LinkedServiceManager(client)
ds_mgr = DatasetManager(client)

# Create Key Vault and ADLS Linked Services
kv_ls = ls_mgr.create_key_vault_linked_service("rg-data-engineering-dev", "adf-data-engineering-dev", "KeyVaultLS", "https://kv-dev.vault.azure.net/")
adls_ls = ls_mgr.create_adls_gen2_linked_service("rg-data-engineering-dev", "adf-data-engineering-dev", "ADLSGen2LS", "https://stdev.dfs.core.windows.net")

# Create Datasets
csv_ds = ds_mgr.create_csv_dataset("rg-data-engineering-dev", "adf-data-engineering-dev", "CsvSourceDS", "ADLSGen2LS", "raw", "sales")
pq_ds = ds_mgr.create_parquet_dataset("rg-data-engineering-dev", "adf-data-engineering-dev", "ParquetSinkDS", "ADLSGen2LS", "gold", "sales")

print("Created Linked Services:", [kv_ls["name"], adls_ls["name"]])
print("Created Datasets:", [csv_ds["name"], pq_ds["name"]])

## Step 2: Deploying & Triggering Copy Activity Pipeline

We now assemble a Copy Activity Pipeline that converts raw CSV files to Parquet columnar format and execute a pipeline run.

In [ ]:
pipeline_mgr = CopyActivityPipelineManager(client)

pipeline_spec = pipeline_mgr.build_blob_to_adls_pipeline_spec(
    pipeline_name="PL_Copy_BlobToADLS",
    source_dataset_name="CsvSourceDS",
    sink_dataset_name="ParquetSinkDS",
    diu_count=8,
    parallel_copies=4
)

deployed = pipeline_mgr.create_or_update_pipeline(
    resource_group="rg-data-engineering-dev",
    factory_name="adf-data-engineering-dev",
    pipeline_name="PL_Copy_BlobToADLS",
    pipeline_spec=pipeline_spec
)

run_id = pipeline_mgr.trigger_pipeline_run(
    resource_group="rg-data-engineering-dev",
    factory_name="adf-data-engineering-dev",
    pipeline_name="PL_Copy_BlobToADLS",
    parameters={"SourceContainer": "raw", "SinkContainer": "gold"}
)

print("Pipeline Deployed:", deployed["name"])
print("Triggered Pipeline Run ID:", run_id)

## Step 3: Monitoring Pipeline Run Execution & Log Analytics KQL Queries

We check the pipeline status, retrieve activity details, and print Log Analytics Kusto Query Language (KQL) queries for production monitoring dashboards.

In [ ]:
monitor = PipelineMonitor(client)

status = monitor.wait_for_completion(
    resource_group="rg-data-engineering-dev",
    factory_name="adf-data-engineering-dev",
    run_id=run_id,
    poll_interval_seconds=1,
    timeout_seconds=30
)

print("Final Pipeline Run Status:", status)

print("\n--- Log Analytics KQL Query for Failed Pipelines ---")
print(monitor.generate_kql_query_failed_pipelines("adf-data-engineering-dev", days_back=7))

print("\n--- Log Analytics KQL Query for Activity Performance ---")
print(monitor.generate_kql_query_activity_performance("adf-data-engineering-dev"))